In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_Narela_Delhi_DPCC_2023.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,291.0,155.0,NaN,105.0,92.0,101.0,59.0,72.0,138.0,152.0,383.0,353.0
1,2,351.0,192.0,249.0,163.0,85.0,97.0,57.0,68.0,157.0,146.0,426.0,319.0
2,3,422.0,223.0,152.0,178.0,116.0,122.0,121.0,64.0,140.0,144.0,476.0,291.0
3,4,364.0,266.0,130.0,115.0,72.0,174.0,132.0,90.0,143.0,174.0,435.0,300.0
4,5,354.0,262.0,199.0,134.0,191.0,167.0,122.0,NaN,123.0,179.0,460.0,304.0
5,6,377.0,304.0,170.0,139.0,247.0,136.0,68.0,NaN,144.0,237.0,435.0,277.0
6,7,374.0,297.0,216.0,145.0,231.0,246.0,65.0,88.0,137.0,280.0,401.0,346.0
7,8,347.0,140.0,237.0,155.0,152.0,161.0,49.0,101.0,111.0,186.0,436.0,372.0
8,9,422.0,212.0,139.0,NaN,216.0,154.0,48.0,119.0,70.0,189.0,426.0,340.0
9,10,385.0,264.0,207.0,209.0,232.0,131.0,NaN,123.0,39.0,NaN,285.0,322.0


In [4]:
df.shape
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        40 non-null     object 
 1   January    36 non-null     float64
 2   February   32 non-null     float64
 3   March      35 non-null     float64
 4   April      32 non-null     float64
 5   May        36 non-null     float64
 6   June       34 non-null     float64
 7   July       25 non-null     float64
 8   August     30 non-null     float64
 9   September  34 non-null     float64
 10  October    35 non-null     float64
 11  November   35 non-null     float64
 12  December   35 non-null     float64
dtypes: float64(12), object(1)
memory usage: 4.3+ KB


In [5]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [6]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [8]:
# Define a function for outlier handling
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            # Replace outliers with mean
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [9]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready.head()

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,291.0,155.0,169.342857,105.0,92.0,101.0,59.00,72.000000,138.0,152.0,383.0,353.0
1,2,351.0,192.0,249.000000,163.0,85.0,97.0,57.00,68.000000,157.0,146.0,426.0,319.0
2,3,422.0,223.0,152.000000,178.0,116.0,122.0,65.24,64.000000,140.0,144.0,476.0,291.0
3,4,364.0,266.0,130.000000,115.0,72.0,174.0,65.24,90.000000,143.0,174.0,435.0,300.0
4,5,354.0,262.0,199.000000,134.0,191.0,167.0,65.24,101.666667,123.0,179.0,460.0,304.0
